# Model Performance: MSE and NLL
Distribution of cross-validated loss (MSE for carrabin and yoo,
NLL for jiang) across participants for each model and task.
One panel per task, 1 row × 3 columns.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve()))
from utils.paths import data_path

In [4]:
sns.set_theme(style='ticks', context='paper', font_scale=1.3)
palette = sns.color_palette("colorblind")
# colors assigned by model role, consistent across tasks:
# optimal (Bayes, Mean) = palette[0]
# naive (RL) = palette[1]
# human-matching (NoisyCounting, DeGroot, ADM) = palette[2]
PALETTE = {
    'Bayes':         palette[0],
    'Mean':          palette[0],
    'RL':            palette[1],
    'NoisyCounting': palette[2],
    'DeGroot':       palette[2],
    'ADM':           palette[2],
}
MODEL_ORDER = {
    'carrabin': ['Bayes', 'RL', 'NoisyCounting'],
    'jiang':    ['Bayes', 'RL', 'DeGroot'],
    'yoo':      ['Mean', 'RL', 'ADM'],
}
YLABELS = {
    'carrabin': 'MSE',
    'jiang':    'NLL',
    'yoo':      'MSE',
}
TITLES = {
    'carrabin': 'Ratio Estimation',
    'jiang':    'Social Learning',
    'yoo':      'Value Comparison',
}

In [ ]:
perf_parts: list[pd.DataFrame] = []
for dataset in MODEL_ORDER:
    for model_type in MODEL_ORDER[dataset]:
        fp = data_path(f"{model_type}_{dataset}_performance.pkl")
        if not fp.exists():
            print(f"Warning: missing collected performance file: {fp}")
            continue
        df = pd.read_pickle(fp)
        perf_parts.append(df[["model_type", "dataset", "pid", "cv_loss_mean"]])

if perf_parts:
    perf = pd.concat(perf_parts, ignore_index=True)
else:
    perf = pd.DataFrame(
        columns=["model_type", "dataset", "pid", "cv_loss_mean"]
    )

print("perf shape:", perf.shape)
combos = perf[["model_type", "dataset"]].drop_duplicates().sort_values(
    ["dataset", "model_type"]
)
print("model_type × dataset combinations:")
print(combos.to_string(index=False))

In [ ]:
proj = Path("..").resolve()
fig_dir = proj / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), constrained_layout=True)
for ax, dataset in zip(axes, ["carrabin", "jiang", "yoo"]):
    subset = perf[perf["dataset"] == dataset]
    order = MODEL_ORDER[dataset]
    if subset.empty:
        ax.set_title(TITLES[dataset])
        ax.text(
            0.5,
            0.5,
            "no data",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        sns.despine(ax=ax, top=True, right=True)
        continue
    sns.violinplot(
        data=subset,
        x="model_type",
        y="cv_loss_mean",
        order=order,
        palette=PALETTE,
        inner="point",
        ax=ax,
    )
    sns.stripplot(
        data=subset,
        x="model_type",
        y="cv_loss_mean",
        order=order,
        color="0.2",
        alpha=0.5,
        jitter=0.2,
        size=4,
        ax=ax,
    )
    ax.set_title(TITLES[dataset])
    ax.set_ylabel(YLABELS[dataset])
    ax.set_xlabel("")
    sns.despine(ax=ax, top=True, right=True)

out_path = fig_dir / "performance_mse_nll.png"
plt.savefig(out_path, dpi=300)
plt.show()

## Notes
- Add observations here as fits complete